#### Setup & Import

In [ ]:
import torch
import torch.nn as nn
import torchvision
import numpy as np
import torchvision.models as models
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
import random
import pandas
import random

from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score

In [ ]:
random_state = 20250520
torch.manual_seed(random_state)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#### Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_from_disk

train_ds = load_from_disk("/content/drive/MyDrive/2025/ML 2/saved_datasets/food101/train")
test_ds = load_from_disk("/content/drive/MyDrive/2025/ML 2/saved_datasets/food101/validation")

print(train_ds)
print(test_ds)

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


Dataset({
    features: ['image', 'label'],
    num_rows: 75750
})
Dataset({
    features: ['image', 'label'],
    num_rows: 25250
})


In [ ]:
classes = train_ds.features['label'].names

#### Transform

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
# Create a wrapper class
class FoodDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        label = item["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
train_ds = train_ds.add_column("idx", list(range(len(train_ds))))
df = train_ds.remove_columns("image").to_pandas()
df_sampled = df.groupby("label", group_keys=False).sample(n=100, random_state=random_state)
sampled_ids = df_sampled["idx"].tolist()
train_sample = train_ds.select(sampled_ids).remove_columns("idx")

# Wrap into PyTorch datasets
train_dataset = FoodDataset(train_sample, transform=transform)
test_dataset = FoodDataset(test_ds, transform=transform)  # Full validation set

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1387: FutureWarning: promote has been superseded by promote_options='default'.
  return cls._concat_blocks(pa_tables_to_concat_vertically, axis=0)


#### Load Pretrained EfficientNet (B0)

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace classifier head
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 101)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 68.6MB/s]


####  Define Loss and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#### Training Loop

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

#### Validation Loop

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc

#### Train for few epochs

In [ ]:
import time

best_val = 0
num_epochs = 20

for epoch in range(num_epochs):
    start = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_acc = evaluate(model, test_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_model.pt")

    print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Val Acc={val_acc:.2%}, Time={time.time()-start:.2f}s")

Training: 100%|██████████| 316/316 [02:05<00:00,  2.51it/s]


Epoch 1: Loss=4.1167, Val Acc=32.24%, Time=312.31s


Training: 100%|██████████| 316/316 [01:46<00:00,  2.97it/s]


Epoch 2: Loss=2.8210, Val Acc=47.70%, Time=292.12s


Training: 100%|██████████| 316/316 [01:47<00:00,  2.93it/s]


...

Epoch 18: Loss=0.1068, Val Acc=58.85%, Time=288.82s


Training: 100%|██████████| 316/316 [01:46<00:00,  2.98it/s]


Epoch 19: Loss=0.0873, Val Acc=59.06%, Time=287.68s


Training: 100%|██████████| 316/316 [01:45<00:00,  2.99it/s]


Epoch 20: Loss=0.0812, Val Acc=58.27%, Time=287.29s


#### confusion matrix, sensitivity value and accuracy for Training Dataset

In [ ]:
predicted_list = []
labels_list = []
with torch.no_grad():
   for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      predicted_list.extend(predicted.tolist())
      labels_list.extend(labels.tolist())

confusion_matrix_train = pandas.crosstab(labels_list, predicted_list)
confusion_matrix_train.columns = classes
confusion_matrix_train.index = classes

n_total = len(labels_list)
n_correct = numpy.sum([1 if labels_list[i] == predicted_list[i] else 0 for i in range(n_total)])

train_accuracy = n_correct / n_total
print(f'Training Accuracy = {train_accuracy:.7f}')

Training Accuracy = 0.9999010


In [ ]:
confusion_matrix_train

,apple_pie,baby_back_ribs,baklava,beef_carpaccio,beef_tartare,beet_salad,beignets,bibimbap,bread_pudding,breakfast_burrito,...,spaghetti_carbonara,spring_rolls,steak,strawberry_shortcake,sushi,tacos,takoyaki,tiramisu,tuna_tartare,waffles
apple_pie,100,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
baby_back_ribs,0,100,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
baklava,0,0,100,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
beef_carpaccio,0,0,0,100,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
beef_tartare,0,0,0,0,100,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tacos,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,100,0,0,0,0
takoyaki,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,100,0,0,0
tiramisu,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,100,0,0
tuna_tartare,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,100,0


In [ ]:
# Calculate sensitivity for each class
sensitivity = {}
for class_label in classes:
    tp = confusion_matrix_train.loc[class_label, class_label]
    fn = confusion_matrix_train.loc[class_label, :].sum() - tp
    sensitivity[class_label] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
print("\nSensitivity (per class):")
for cls, val in sensitivity.items():
    print(f"{cls}: {val:.4f}")


Sensitivity (per class):
apple_pie: 1.0000
baby_back_ribs: 1.0000
baklava: 1.0000
beef_carpaccio: 1.0000
beef_tartare: 1.0000
beet_salad: 1.0000
beignets: 1.0000
bibimbap: 1.0000
bread_pudding: 1.0000
breakfast_burrito: 1.0000
bruschetta: 1.0000
caesar_salad: 1.0000
cannoli: 1.0000
caprese_salad: 1.0000
carrot_cake: 1.0000
ceviche: 1.0000
cheesecake: 1.0000
cheese_plate: 1.0000
chicken_curry: 1.0000
chicken_quesadilla: 1.0000
chicken_wings: 1.0000
chocolate_cake: 1.0000
chocolate_mousse: 1.0000
churros: 1.0000
clam_chowder: 1.0000
club_sandwich: 1.0000
crab_cakes: 1.0000
creme_brulee: 1.0000
croque_madame: 1.0000
cup_cakes: 1.0000
deviled_eggs: 1.0000
donuts: 1.0000
dumplings: 1.0000
edamame: 1.0000
eggs_benedict: 1.0000
escargots: 1.0000
falafel: 1.0000
filet_mignon: 1.0000
fish_and_chips: 1.0000
foie_gras: 1.0000
french_fries: 1.0000
french_onion_soup: 1.0000
french_toast: 1.0000
fried_calamari: 1.0000
fried_rice: 1.0000
frozen_yogurt: 1.0000
garlic_bread: 1.0000
gnocchi: 1.0000
gre

#### confusion matrix, sensitivity value and accuracy for Testing Dataset

In [ ]:
PATH = './best_model.pt'
torch.save(model.state_dict(), PATH)

In [ ]:
# Load the model back
#model = Net()
model.to(device)
model.load_state_dict(torch.load(PATH, weights_only=True))

predicted_list = []
labels_list = []
with torch.no_grad():
   for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      predicted_list.extend(predicted.tolist())
      labels_list.extend(labels.tolist())

confusion_matrix_test = pandas.crosstab(labels_list, predicted_list)
confusion_matrix_test.columns = classes
confusion_matrix_test.index = classes

n_total = len(labels_list)
n_correct = numpy.sum([1 if labels_list[i] == predicted_list[i] else 0 for i in range(n_total)])

accuracy = n_correct / n_total
print(f'Testing Accuracy = {accuracy:.7f}')

Testing Accuracy = 0.5826931


In [ ]:
confusion_matrix_test

,apple_pie,baby_back_ribs,baklava,beef_carpaccio,beef_tartare,beet_salad,beignets,bibimbap,bread_pudding,breakfast_burrito,...,spaghetti_carbonara,spring_rolls,steak,strawberry_shortcake,sushi,tacos,takoyaki,tiramisu,tuna_tartare,waffles
apple_pie,51,1,7,0,1,0,7,0,12,4,...,1,0,0,2,2,0,0,0,2,11
baby_back_ribs,1,172,0,0,0,0,0,1,0,1,...,0,0,9,0,0,0,0,0,1,1
baklava,6,2,157,0,0,1,2,0,3,0,...,0,2,0,0,1,0,1,0,3,3
beef_carpaccio,0,0,0,159,1,11,0,0,1,0,...,0,0,2,2,0,1,2,0,2,2
beef_tartare,2,3,1,1,120,6,1,2,2,0,...,0,0,5,0,1,1,1,2,13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tacos,0,1,0,6,1,1,1,3,0,7,...,1,3,0,0,3,104,10,0,0,0
takoyaki,0,0,1,4,1,9,2,0,0,0,...,0,0,0,0,4,1,148,1,1,0
tiramisu,1,0,2,1,0,0,2,0,6,0,...,0,0,0,4,0,0,0,132,0,2
tuna_tartare,6,1,0,9,27,11,0,0,3,2,...,0,0,2,2,2,2,1,0,78,0


In [ ]:
# Calculate sensitivity for each class
sensitivity = {}
for class_label in classes:
    tp = confusion_matrix_test.loc[class_label, class_label]
    fn = confusion_matrix_test.loc[class_label, :].sum() - tp
    sensitivity[class_label] = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# Display sensitivity
print("\nSensitivity (per class):")
for cls, val in sensitivity.items():
    print(f"{cls}: {val:.4f}")


Sensitivity (per class):
apple_pie: 0.2040
baby_back_ribs: 0.6880
baklava: 0.6280
beef_carpaccio: 0.6360
beef_tartare: 0.4800
beet_salad: 0.4680
beignets: 0.7240
bibimbap: 0.7840
bread_pudding: 0.3040
breakfast_burrito: 0.3840
bruschetta: 0.3680
caesar_salad: 0.5680
cannoli: 0.6200
caprese_salad: 0.4440
carrot_cake: 0.6080
ceviche: 0.2840
cheesecake: 0.5280
cheese_plate: 0.4120
chicken_curry: 0.4040
chicken_quesadilla: 0.5680
chicken_wings: 0.7560
chocolate_cake: 0.4960
chocolate_mousse: 0.3560
churros: 0.7200
clam_chowder: 0.8320
club_sandwich: 0.5800
crab_cakes: 0.4480
creme_brulee: 0.7080
croque_madame: 0.6560
cup_cakes: 0.8200
deviled_eggs: 0.7280
donuts: 0.6600
dumplings: 0.8120
edamame: 0.9480
eggs_benedict: 0.6560
escargots: 0.7040
falafel: 0.5480
filet_mignon: 0.5200
fish_and_chips: 0.6840
foie_gras: 0.3240
french_fries: 0.8200
french_onion_soup: 0.6080
french_toast: 0.4720
fried_calamari: 0.6360
fried_rice: 0.6960
frozen_yogurt: 0.8000
garlic_bread: 0.4600
gnocchi: 0.5520
gre